In [ ]:
from google.colab import files
import os

# Check if the file already exists to avoid re-uploading every time
if not os.path.exists('crimes_cleaned.csv'):
  print('Please upload the "crimes_cleaned.csv" file:')
  uploaded = files.upload()
  if 'crimes_cleaned.csv' not in uploaded:
    print('"crimes_cleaned.csv" was not uploaded. Please upload the file to proceed.')
else:
  print('"crimes_cleaned.csv" already exists. Skipping upload.')

Please upload the "crimes_cleaned.csv" file:


Saving crimes_cleaned.csv to crimes_cleaned.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# Load data
df = pd.read_csv("crimes_cleaned.csv")

# Keep only rows with valid coordinates
cluster_df = df[['latitude', 'longitude', 'primary_type', 'district', 'location_description']].dropna().copy()

# DBSCAN works best for geo coordinates in radians with haversine metric
coords = np.radians(cluster_df[['latitude', 'longitude']].values)

# Earth radius in kilometers
earth_radius_km = 6371.0088

# eps = neighborhood radius in km converted to radians
# Start with 0.5 km; tune between 0.3 and 1.0
eps_km = 0.2
eps = eps_km / earth_radius_km

# Fit DBSCAN
db = DBSCAN(
    eps=eps, # Use the calculated 'eps' value (in radians)
    min_samples=10,
    metric='haversine',
    algorithm='ball_tree'
)

cluster_df['cluster'] = db.fit_predict(coords)

# Save clustered points for Power BI
cluster_df.to_csv("crime_clusters_for_powerbi.csv", index=False)

# Create cluster summary
summary = (
    cluster_df[cluster_df['cluster'] != -1]
    .groupby('cluster')
    .size()
    .reset_index(name='crime_count')
    .sort_values('crime_count', ascending=False)
)

summary.to_csv("cluster_summary_for_powerbi.csv", index=False)

# Top crime types per cluster
top_types = (
    cluster_df[cluster_df['cluster'] != -1]
    .groupby(['cluster', 'primary_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['cluster', 'count'], ascending=[True, False])
)

top_types.to_csv("cluster_top_types_for_powerbi.csv", index=False)

print(cluster_df['cluster'].value_counts().head(10))

cluster
 0     49877
-1      3555
 3      2008
 8       518
 25      275
 1       219
 16      213
 37      194
 28      170
 13      136
Name: count, dtype: int64


In [ ]:
from google.colab import files

print('Downloading generated files...')
files_to_download = [
    'crime_clusters_for_powerbi.csv',
    'cluster_summary_for_powerbi.csv',
    'cluster_top_types_for_powerbi.csv'
]

for filename in files_to_download:
    if os.path.exists(filename):
        files.download(filename)
        print(f'Downloaded: {filename}')
    else:
        print(f'File not found, skipping download: {filename}')

print('Download process complete.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: crime_clusters_for_powerbi.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: cluster_summary_for_powerbi.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: cluster_top_types_for_powerbi.csv
Download process complete.
